# Cluster items into synthetic categories

Issue: #8

## Question

OTTO has no categories. Can we make 50 to 100 stable, reasonably balanced ones from behaviour alone?

## Data used

Input: `data/covis/covis.npz`, `item_ids.npy`, `meta.json` from #7. That matrix was built from the training window w0 only, and this notebook checks that.

Output in `data/categories/`: `item_category.parquet` (columns `aid`, `cluster`, one row per assigned item), `centroids.npy`, `item_embedding.npy`. All categories are **synthetic**: they come from behaviour and have no real meaning. Names come in #9.

## Method and the rule for choosing K (written before looking at results)

1. Transform weights with `log1p`, reduce the co-visitation matrix with truncated SVD to 64 dimensions, and scale each item vector to unit length (so k-means groups by direction, like cosine similarity).
2. Cluster with k-means (MiniBatch) for K in 30, 40, 50, 60, 70, 80, 90, 100, 120, 150, three seeds each.
3. **Stability** of a K = mean adjusted Rand index (ARI) between the labelings of different seeds. ARI ignores label names, so it is safe to compare across seeds.
4. **Size balance** = share of items in the largest cluster, and the number of clusters with fewer than 500 items.
5. **Rule.** Among K from 50 to 100 whose largest cluster holds at most 5 percent of items, choose the K with the highest stability. If none qualify, choose the K with the smallest largest-cluster share.
6. Compare with Leiden community detection on the same two criteria.
7. After choosing, merge any cluster with fewer than 500 items into the nearest larger cluster.

In [ ]:
import json
import resource
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import sparse
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
COVIS = DATA / "covis"
OUT = DATA / "categories"
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = [0, 1, 2]
MIN_CLUSTER = 500


def peak_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9


matrix = sparse.load_npz(COVIS / "covis.npz")
item_ids = np.load(COVIS / "item_ids.npy")
meta = json.loads((COVIS / "meta.json").read_text())
assert meta["train_end_ms"] == day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"]), "matrix is not from the training window"
print("items:", len(item_ids), " nnz:", matrix.nnz, " training window ends at", meta["train_end_ms"])

## Embed the items

In [ ]:
start = time.time()
log_matrix = matrix.copy()
log_matrix.data = np.log1p(log_matrix.data)
svd = TruncatedSVD(n_components=64, n_iter=5, random_state=0)
embedding = svd.fit_transform(log_matrix).astype(np.float32)
norms = np.linalg.norm(embedding, axis=1)
has_vector = norms > 1e-6
print(f"SVD done in {time.time() - start:.0f} s; explained variance {svd.explained_variance_ratio_.sum():.1%}")
print(f"items with no co-visits (zero vector, left unassigned): {(~has_vector).sum():,}")
embedding[has_vector] /= norms[has_vector, None]
usable = np.flatnonzero(has_vector)          # positions in item_ids that get a category
X = embedding[usable]
print("items to cluster:", len(usable))

## k-means over a range of K

In [ ]:
def size_stats(labels):
    sizes = np.bincount(labels)
    return {
        "k_found": int((sizes > 0).sum()),
        "largest_share": float(sizes.max() / sizes.sum()),
        "n_small": int(((sizes > 0) & (sizes < MIN_CLUSTER)).sum()),
        "min_size": int(sizes[sizes > 0].min()),
    }


def stability(label_list):
    pairs = [(a, b) for a in range(len(label_list)) for b in range(a + 1, len(label_list))]
    return float(np.mean([adjusted_rand_score(label_list[a], label_list[b]) for a, b in pairs]))


rows = []
labels_by_k = {}
for k in [30, 40, 50, 60, 70, 80, 90, 100, 120, 150]:
    start = time.time()
    runs = [
        MiniBatchKMeans(n_clusters=k, n_init=3, batch_size=20_000, random_state=s).fit_predict(X)
        for s in SEEDS
    ]
    labels_by_k[k] = runs[0]
    stats = size_stats(runs[0])
    rows.append({"K": k, "stability_ari": stability(runs), **stats})
    print(f"K={k}: ARI {rows[-1]['stability_ari']:.3f}, largest {stats['largest_share']:.1%}, small {stats['n_small']} ({time.time() - start:.0f} s)")
kmeans_table = pl.DataFrame(rows)
print(kmeans_table)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
ax1.plot(kmeans_table["K"], kmeans_table["stability_ari"], "o-")
ax1.axvspan(50, 100, color="green", alpha=0.08)
ax1.set_xlabel("K"); ax1.set_ylabel("stability (mean ARI across seeds)"); ax1.set_title("Stability vs K")
ax2.plot(kmeans_table["K"], kmeans_table["largest_share"] * 100, "o-", label="largest cluster %")
ax2.plot(kmeans_table["K"], kmeans_table["n_small"], "s--", label=f"clusters < {MIN_CLUSTER} items")
ax2.axvspan(50, 100, color="green", alpha=0.08)
ax2.set_xlabel("K"); ax2.set_title("Size balance vs K"); ax2.legend()
plt.tight_layout(); plt.show()

## Choose K by the rule

In [ ]:
candidates = kmeans_table.filter((pl.col("K") >= 50) & (pl.col("K") <= 100))
ok = candidates.filter(pl.col("largest_share") <= 0.05)
if ok.height:
    best = ok.sort("stability_ari", descending=True).row(0, named=True)
    how = "highest stability among K in 50-100 with largest cluster <= 5%"
else:
    best = candidates.sort("largest_share").row(0, named=True)
    how = "no K qualified, smallest largest-cluster share"
K = best["K"]
print(f"chosen K = {K} ({how})")
print(best)
assert 50 <= K <= 100

## Alternative: Leiden community detection on the co-visitation graph

The graph keeps each item's 20 strongest neighbours (undirected, weights summed) so that it fits in memory. The resolution parameter controls how many communities come out, so it is tuned to land between 50 and 100.

In [ ]:
import igraph as ig
import leidenalg

start = time.time()
top = 20
keep_cols, keep_rows, keep_w = [], [], []
csr = matrix.tocsr()
for r in range(csr.shape[0]):
    a, b = csr.indptr[r], csr.indptr[r + 1]
    if a == b:
        continue
    d = csr.data[a:b]
    idx = np.argpartition(d, -top)[-top:] if len(d) > top else np.arange(len(d))
    keep_rows.append(np.full(len(idx), r)); keep_cols.append(csr.indices[a:b][idx]); keep_w.append(d[idx])
rows_ = np.concatenate(keep_rows); cols_ = np.concatenate(keep_cols); w_ = np.concatenate(keep_w).astype(float)
sym = sparse.coo_matrix((w_, (np.minimum(rows_, cols_), np.maximum(rows_, cols_))), shape=csr.shape).tocsr()
sym.sum_duplicates()
sym = sym.tocoo()
graph = ig.Graph(n=csr.shape[0], edges=list(zip(sym.row.tolist(), sym.col.tolist())), directed=False)
graph.es["weight"] = sym.data.tolist()
print(f"graph: {graph.vcount():,} nodes, {graph.ecount():,} edges, built in {time.time() - start:.0f} s")
del rows_, cols_, w_, keep_rows, keep_cols, keep_w

In [ ]:
def leiden(resolution, seed):
    part = leidenalg.find_partition(
        graph, leidenalg.RBConfigurationVertexPartition, weights="weight",
        resolution_parameter=resolution, n_iterations=2, seed=seed,
    )
    return np.array(part.membership)


tuning = []
for res in [0.05, 0.3, 1.0, 3.0]:
    start = time.time()
    m = leiden(res, 0)[usable]
    _, m = np.unique(m, return_inverse=True)
    st = size_stats(m)
    tuning.append((res, len(np.unique(m)), st["largest_share"]))
    print(f"resolution {res}: {tuning[-1][1]} communities, largest {st['largest_share']:.1%}, small {st['n_small']} ({time.time() - start:.0f} s)")

In [ ]:
# Count alone is a bad criterion (one giant community plus tiny ones can still give 50 to 100).
# Rule: among resolutions whose largest community is at most 20% of items, take the count closest to 75;
# if none qualify, take the smallest largest-share.
good = [t for t in tuning if t[2] <= 0.20]
res = (min(good, key=lambda t: abs(t[1] - 75)) if good else min(tuning, key=lambda t: t[2]))[0]
runs = []
for s in SEEDS:
    m = leiden(res, s)[usable]
    _, m = np.unique(m, return_inverse=True)
    runs.append(m)
leiden_stats = {"method": f"Leiden (resolution {res})", "K": len(np.unique(runs[0])), "stability_ari": stability(runs), **size_stats(runs[0])}
km_stats = {"method": "k-means", "K": K, "stability_ari": best["stability_ari"], **size_stats(labels_by_k[K])}
comparison = pl.DataFrame([km_stats, leiden_stats]).select("method", "K", "stability_ari", "largest_share", "n_small", "min_size")
print(comparison)

## Final assignment with k-means

Clusters smaller than 500 items are merged into the nearest larger cluster (cosine similarity to its centroid).

In [ ]:
labels = labels_by_k[K].copy()
sizes = np.bincount(labels, minlength=K)
small = np.flatnonzero(sizes < MIN_CLUSTER)
centroids = np.stack([X[labels == c].mean(axis=0) for c in range(K)])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
if len(small):
    big = np.setdiff1d(np.arange(K), small)
    for c in small:
        members = labels == c
        labels[members] = big[np.argmax(X[members] @ centroids[big].T, axis=1)]
    print(f"merged {len(small)} small cluster(s): sizes {sizes[small].tolist()}")
else:
    print("no cluster below the size limit, nothing merged")
_, labels = np.unique(labels, return_inverse=True)
K_final = int(labels.max() + 1)
centroids = np.stack([X[labels == c].mean(axis=0) for c in range(K_final)])
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
final_sizes = np.bincount(labels)
print(f"final: {K_final} clusters, sizes min {final_sizes.min():,} median {int(np.median(final_sizes)):,} max {final_sizes.max():,}")

table = pl.DataFrame({"aid": item_ids[usable].astype(np.int32), "cluster": labels.astype(np.int16)})
table.write_parquet(OUT / "item_category.parquet")
np.save(OUT / "centroids.npy", centroids)
np.save(OUT / "item_embedding.npy", embedding)
print(table.head())

In [ ]:
check = pl.read_parquet(OUT / "item_category.parquet")
assert check["aid"].n_unique() == check.height, "duplicate items"
assert check["aid"].is_in(item_ids.tolist()).all()
assert final_sizes.min() >= MIN_CLUSTER
assert 50 <= K_final <= 100, K_final
print(f"{check.height:,} items assigned, {check['cluster'].n_unique()} clusters, no duplicates, smallest cluster {final_sizes.min():,}")
print(f"{len(item_ids) - check.height:,} items in the matrix have no co-visits and stay unassigned; items below the frequency threshold are not in the matrix and are also unassigned")
print(f"peak memory: {peak_gb():.2f} GB")
assert peak_gb() < 7

## Result

See the two plots, the k-means table, the chosen K with its reason, and the comparison table for Leiden. The output file has one row per assigned item, built from the training window only. Categories are synthetic.

Notes to carry forward:
- 64 SVD dimensions explain only about 5 percent of the variance of the (very sparse) matrix. That is normal for co-visitation data, but it means the embedding is a coarse summary.
- Leiden was tuned twice: the first attempt used the number of communities as the only criterion and produced one giant community holding 99.97 percent of items, so the tuning rule now also requires a balanced largest community. See the tuning printout for what each resolution gave.

**Verdict (k-means vs Leiden).** Once Leiden's resolution was tuned properly (resolution 1.0, 78 communities), the two methods are equally stable (ARI 0.772 vs 0.770). K-means is far better balanced: its largest cluster holds 3.0 percent of items and none has fewer than 1,053 items, while Leiden's largest holds 10.1 percent and 35 of its 78 communities have fewer than 500 items (some a single item). K-means with K = 80 is used. The Leiden resolution grid was coarse (0.05, 0.3, 1.0, 3.0), so a finer search might do better.

## What I learned

To be written by the owner of the project.